In [6]:
import pandas as pd
import os

# Set working directory to project root
os.chdir(r'C:\10x AIMastery\fraud-detection-10academy')


merged = pd.read_csv("data/processed/fraud_data_with_country.csv")  # or your actual path


In [7]:
from sklearn.model_selection import train_test_split

# Drop unneeded columns and define X, y
features = merged.drop(columns=['user_id', 'device_id', 'signup_time', 'purchase_time', 'class'])
target = merged['class']

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, stratify=target, random_state=42)


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Identify feature types
num_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X_train.select_dtypes(include=['object']).columns.tolist()

# Pipelines
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Column transformer
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Logistic Regression Pipeline
lr_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Random Forest Pipeline
rf_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])


In [6]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import seaborn as sns
import matplotlib.pyplot as plt

# Create output directory if it doesn't exist
os.makedirs('output', exist_ok=True)
os.makedirs('data', exist_ok=True)

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import seaborn as sns
import matplotlib.pyplot as plt

# Load datasets
fraud_data = pd.read_csv(r'C:\10x AIMastery\fraud-detection-10academy\data\raw\Fraud_Data.csv')
ip_to_country = pd.read_csv(r'C:\10x AIMastery\fraud-detection-10academy\data\raw\IpAddress_to_Country.csv')
creditcard_data = pd.read_csv(r'C:\10x AIMastery\fraud-detection-10academy\data\raw\creditcard.csv')

# Step 1: Handle Missing Values
# Impute numerical and categorical columns
imputer_num = SimpleImputer(strategy='median')
imputer_cat = SimpleImputer(strategy='most_frequent')

# Fraud_Data.csv
num_cols = ['purchase_value', 'age']
fraud_data[num_cols] = imputer_num.fit_transform(fraud_data[num_cols])
cat_cols = ['source', 'browser', 'sex']
fraud_data[cat_cols] = imputer_cat.fit_transform(fraud_data[cat_cols])
fraud_data = fraud_data.dropna(subset=['class'])

# creditcard.csv
creditcard_data = creditcard_data.dropna(subset=['Class'])

# Step 2: Data Cleaning
# Remove duplicates
fraud_data = fraud_data.drop_duplicates()
creditcard_data = creditcard_data.drop_duplicates()

# Correct data types
fraud_data['signup_time'] = pd.to_datetime(fraud_data['signup_time'])
fraud_data['purchase_time'] = pd.to_datetime(fraud_data['purchase_time'])
fraud_data['ip_address'] = fraud_data['ip_address'].astype(int)

# Step 3: Exploratory Data Analysis (EDA)
# Univariate: Distribution of purchase_value
plt.figure(figsize=(10, 6))
sns.histplot(fraud_data['purchase_value'], bins=50)
plt.title('Distribution of Purchase Value')
plt.savefig('output/purchase_value_dist.png')
plt.close()

# Bivariate: Fraud rate by browser
fraud_by_browser = fraud_data.groupby('browser')['class'].mean()
plt.figure(figsize=(10, 6))
sns.barplot(x=fraud_by_browser.index, y=fraud_by_browser.values)
plt.title('Fraud Rate by Browser')
plt.savefig('output/fraud_by_browser.png')
plt.close()

# Step 4: Merge Datasets for Geolocation
ip_to_country['lower_bound_ip_address'] = ip_to_country['lower_bound_ip_address'].astype(int)
ip_to_country['upper_bound_ip_address'] = ip_to_country['upper_bound_ip_address'].astype(int)

def map_ip_to_country(ip):
    match = ip_to_country[(ip_to_country['lower_bound_ip_address'] <= ip) & 
                          (ip_to_country['upper_bound_ip_address'] >= ip)]['country']
    return match.iloc[0] if not match.empty else 'Unknown'

fraud_data['country'] = fraud_data['ip_address'].apply(map_ip_to_country)

# Step 5: Feature Engineering
# Transaction frequency and velocity
fraud_data['transaction_count'] = fraud_data.groupby('user_id')['user_id'].transform('count')
fraud_data['avg_time_between_transactions'] = fraud_data.groupby('user_id')['purchase_time'].diff().dt.total_seconds().fillna(0)

# Time-based features
fraud_data['hour_of_day'] = fraud_data['purchase_time'].dt.hour
fraud_data['day_of_week'] = fraud_data['purchase_time'].dt.dayofweek
fraud_data['time_since_signup'] = (fraud_data['purchase_time'] - fraud_data['signup_time']).dt.total_seconds() / 3600

# Step 6: Data Transformation
# Encode categorical features
cat_cols = ['source', 'browser', 'sex', 'country']
encoder = OneHotEncoder(sparse_output=False, drop='first')
encoded_cats = encoder.fit_transform(fraud_data[cat_cols])
encoded_df = pd.DataFrame(encoded_cats, columns=encoder.get_feature_names_out(cat_cols))
fraud_data = pd.concat([fraud_data.drop(cat_cols, axis=1), encoded_df], axis=1)

# Scale numerical features
num_cols = ['purchase_value', 'age', 'transaction_count', 'avg_time_between_transactions', 'time_since_signup']
scaler = StandardScaler()
fraud_data[num_cols] = scaler.fit_transform(fraud_data[num_cols])

# creditcard.csv scaling
creditcard_num_cols = ['Time', 'Amount'] + [f'V{i}' for i in range(1, 29)]
creditcard_data[creditcard_num_cols] = scaler.fit_transform(creditcard_data[creditcard_num_cols])

# Save processed datasets
fraud_data.to_csv('data/processed_fraud_data.csv', index=False)
creditcard_data.to_csv('data/processed_creditcard_data.csv', index=False)

print("Preprocessing complete. Processed datasets saved.")

Preprocessing complete. Processed datasets saved.
